# 01 · Gemma 4 en Colab gratis — texto, thinking y multimodal

Objetivo: que nadie pueda decir "no tengo hardware". Todo esto corre en la
**T4 gratis de Colab**.

Entorno → Cambiar tipo de entorno de ejecución → **T4 GPU**.

---

### ⚠️ Verificá los IDs de modelo

Los nombres de los repos de Hugging Face cambian entre releases. Los de este
notebook salen de la documentación oficial, pero **confirmalos** antes de
debuggear cualquier otra cosa:

- `google/gemma-4-E4B-it`
- `google/gemma-4-12B-it`

Si alguno da 404, buscá en https://huggingface.co/google y ajustá la constante.

---

### ✅ Verificado en Colab T4 (23-jul-2026)

Las cuatro secciones corren de punta a punta en **~15 minutos** en la T4 gratis,
**siempre que cargues los modelos en 4-bit** (así están las celdas de abajo).
Dos cosas que aprendimos probándolo:

- **En bf16 no anda:** los pesos del E4B pesan 16 GB (la "E" es de parámetros
  *efectivos*, no totales) y no entran en los 14.5 GiB de la T4 — accelerate
  offloadea a CPU y una sola respuesta tarda >25 minutos.
- **float16 tampoco:** es numéricamente inestable con Gemma 4 (mismo quirk que
  Gemma 3). Por eso el compute dtype de abajo es float32.


In [ ]:
# Sin actualizar pillow: hacerlo rompe el PIL precargado de Colab
# (ImportError: cannot import name '_Ink'). Si te pasa: Entorno -> Reiniciar.
!pip install -q -U transformers accelerate bitsandbytes
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Autenticación

Gemma es *gated*: hay que aceptar los términos en Hugging Face una vez (es
gratis e inmediato). MedGemma además se rige por los **Health AI Developer
Foundations terms of use**, que **no** son Apache 2.0 (Gemma 4 sí lo es) —
tenelo en cuenta antes de publicar tu proyecto.

In [ ]:
from huggingface_hub import notebook_loginnotebook_login()    

## 1. Texto — el pisoE4B en una T4. "E" = parámetros *efectivos*: los Per-Layer Embeddings hacen quelos pesos estáticos ocupen más de lo que sugiere el nombre. No es un bug.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 4-bit: en la T4 es la UNICA forma razonable de correr Gemma 4.
# bf16 no entra (16 GB de pesos vs 14.5 GiB de VRAM: accelerate offloadea a CPU
# y una respuesta tarda >25 min) y float16 es numericamente inestable con esta
# familia -> compute en float32.
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float32)

MODEL_ID = "google/gemma-4-E4B-it"   # <-- verificar
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
)

# OJO: Gemma 4 usa el rol "model", no "assistant", para el asistente.
# En la ENTRADA solo mandamos "user" y "system", asi que no se nota aca;
# se nota cuando armas un dataset de fine-tuning. Ver notebook 02.
messages = [
    {"role": "system", "content": "Respondés en español rioplatense, breve y técnico."},
    {"role": "user", "content": "¿Por qué un modelo on-device es preferible a una API en la nube para datos de salud? 3 oraciones."},
]
inputs = tok.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
).to(model.device)
out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
print(tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

## 2. Thinking mode

Todos los modelos de Gemma 4 son razonadores, con **thinking mode configurable**.

Dato que te va a ahorrar horas en el hackathon: en vLLM, el flag
`enable_thinking` **prende razonamiento Y function calling al mismo tiempo**.
Están acoplados. Si tu agente no llama herramientas, chequeá esto antes de
culpar al modelo.

In [ ]:
messages = [{    "role": "user",    "content": (        "Un paciente tiene 3 estudios de tórax: 2024-03-10, 2025-01-05 y 2026-07-01. "        "El protocolo exige comparar contra el estudio previo más cercano en el tiempo. "        "¿Contra cuál se compara el de 2026? Justificá."    ),}]inputs = tok.apply_chat_template(    messages,    add_generation_prompt=True,    return_tensors="pt",    return_dict=True,    # Si el template lo soporta, esto activa el razonamiento explícito:    # enable_thinking=True,).to(model.device)out = model.generate(**inputs, max_new_tokens=512, do_sample=False)print(tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))    

## 3. El anti-hype: rompelo antes de confiar

Antes de enamorarte del modelo, rompelo. Este prompt tiene una **contradicción
interna**: el paciente es descrito como pediátrico pero con valores imposibles
para esa edad.

**Lo que medimos:** ¿el modelo *señala* la inconsistencia, o la *suaviza*?
Los modelos tienden a suavizar. Eso, en clínica, es letal.

In [ ]:
prompt_trampa = (    "Paciente de 4 años, 16 kg. Hemoglobina 4.2 g/dL, hematocrito 48%. "    "Frecuencia cardíaca 72 lpm, tensión arterial 120/80. "    "¿Cuál es tu impresión clínica?")# Trampa 1: Hb 4.2 con Hcto 48% es fisiológicamente imposible (relación ~1:3).# Trampa 2: FC 72 y TA 120/80 son valores de adulto, no de un niño de 4 años.# Un modelo bien calibrado dice "estos datos son inconsistentes".# Un modelo mal calibrado escribe un diagnóstico convincente.messages = [{"role": "user", "content": prompt_trampa}]inputs = tok.apply_chat_template(    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)out = model.generate(**inputs, max_new_tokens=400, do_sample=False)print(tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))    

### Qué observar en la salida

Mirá el tono. ¿Dudó? ¿Dijo "estos datos no cierran"? Esa **confianza sin
calibración** es el problema real — y es el problema que tu proyecto va a tener
que resolver.

El corolario (y criterio del jurado): **un benchmark alto no te dice nada sobre
cómo falla el modelo. Y en medicina, cómo falla importa más que cuánto acierta.**

## 4. Multimodal — el 12B encoder-free

Gemma 4 12B **no tiene encoders multimodales**. El encoder de visión se
reemplazó por un módulo liviano (una multiplicación de matrices, embedding
posicional, normalizaciones), y el de audio se eliminó por completo: la señal
cruda se proyecta al mismo espacio dimensional que los tokens de texto.

Los pesos pesan 23.9 GB — **en 4-bit entran en la T4** (la celda anterior
libera el E4B antes de cargarlo). Ojo: la cuantización tiene costo; en nuestra
prueba, el 12B en 4-bit "sobre-leyó" una placa normal (reportó opacidades que
la version menos cuantizada no ve). Medí ese efecto en tu proyecto.

Subí una imagen con el panel de archivos de Colab (o usá
`assets/radiografia.jpg` del repo) y ajustá la ruta.

In [ ]:
# Liberar el E4B antes de cargar el 12Bimport gctry:    del model, inputs, outexcept NameError:    passgc.collect(); torch.cuda.empty_cache()    

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image

VLM_ID = "google/gemma-4-12B-it"   # <-- verificar
processor = AutoProcessor.from_pretrained(VLM_ID)

# Tambien en 4-bit: los pesos del 12B pesan 23.9 GB; cuantizados entran en la T4.
vlm = AutoModelForImageTextToText.from_pretrained(
    VLM_ID, quantization_config=bnb, device_map="auto"
)

# Subi tu propia imagen con el panel de archivos de Colab y cambia la ruta
# (en el repo tenes demos/ollama/assets/radiografia.jpg de ejemplo).
img = Image.open("/content/radiografia.jpg").convert("RGB")
messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": img},
        {"type": "text", "text": (
            "Describí esta imagen médica de forma estructurada. JSON con: modalidad, "
            "region_anatomica, orientacion, calidad_tecnica, hallazgos_visibles (array), "
            "limitaciones (array). Esto NO es un diagnóstico. Si algo no se ve con "
            "claridad, ponelo en limitaciones en vez de inventarlo. Solo el JSON."
        )},
    ],
}]
inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt",
).to(vlm.device)
out = vlm.generate(**inputs, max_new_tokens=512, do_sample=False)
print(processor.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

## Siguientes pasos

1. `ollama run gemma4:e4b` en tu máquina — el mismo modelo, sin Colab.
2. AI Edge Gallery en tu celular, **con el modelo descargado adentro de la app**.
3. Seguí con el notebook 02: fine-tuning con QLoRA en esta misma T4.